In [12]:
import numpy as np 
import pandas as pd

In [13]:
sample_df = pd.read_csv('sample_customer_data.csv')

In [14]:
sample_df.head()

,customer_name,region,age,signup_date,purchase_amount,product_category,satisfaction_score
0,Customer_1,South,NaN,2022-01-01 00:00:00.000000,129.64,Clothing,3.0
1,Customer_2,NaN,69.0,2022-01-06 12:03:37.085427,101.08,Groceries,3.0
2,Customer_3,West,28.0,2022-01-12 00:07:14.170854,191.95,Groceries,4.0
3,Customer_4,East,66.0,2022-01-17 12:10:51.256281,185.99,Groceries,3.0
4,Customer_5,North,25.0,2022-01-23 00:14:28.341708,NaN,Clothing,4.0


In [15]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_name       208 non-null    str    
 1   region              199 non-null    str    
 2   age                 197 non-null    float64
 3   signup_date         208 non-null    str    
 4   purchase_amount     196 non-null    float64
 5   product_category    192 non-null    str    
 6   satisfaction_score  199 non-null    float64
dtypes: float64(3), str(4)
memory usage: 11.5 KB


In [16]:
sample_df.isnull().sum()

customer_name          0
region                 9
age                   11
signup_date            0
purchase_amount       12
product_category      16
satisfaction_score     9
dtype: int64

In [17]:
sample_df.duplicated().sum()

np.int64(8)

In [18]:
# 1. Remove exact duplicate rows
sample_df = sample_df.drop_duplicates()

In [36]:
# 2. Standardize text columns: strip whitespace, fix casing
sample_df['customer_name'] = sample_df['customer_name'].str.strip().str.title()
sample_df['region'] = sample_df['region'].str.strip().str.title()
sample_df['product_category'] = sample_df['product_category'].str.strip().str.title()

In [20]:
 # 3. Fix invalid ages (negative values -> absolute value; unrealistic ages -> NaN)
sample_df['age'] = sample_df['age'].abs()
sample_df.loc[(sample_df['age']>100), 'age'] = np.nan
sample_df

,customer_name,region,age,signup_date,purchase_amount,product_category,satisfaction_score
0,Customer_1,South,NaN,2022-01-01 00:00:00.000000,129.64,Clothing,3.0
1,Customer_2,NaN,69.0,2022-01-06 12:03:37.085427,101.08,Groceries,3.0
2,Customer_3,West,28.0,2022-01-12 00:07:14.170854,191.95,Groceries,4.0
3,Customer_4,East,66.0,2022-01-17 12:10:51.256281,185.99,Groceries,3.0
4,Customer_5,North,25.0,2022-01-23 00:14:28.341708,NaN,Clothing,4.0
...,...,...,...,...,...,...,...
195,Customer_196,South,29.0,2024-12-08 23:45:31.658291,185.17,Groceries,4.0
196,Customer_197,West,43.0,2024-12-14 11:49:08.743718,58.61,Home,4.0
197,Customer_198,north,33.0,2024-12-19 23:52:45.829145,3335.25,Groceries,3.0
198,Customer_199,north,68.0,2024-12-25 11:56:22.914572,172.54,Home,3.0


In [ ]:
# 4. Handle outliers in purchase_amount using the IQR method
q1 = sample_df['purchase_amount'].quantile(0.25)
q3 = sample_df['purchase_amount'].quantile(0.75)
iqr = q3-q1
lower_bound = q1 - 1.5*iqr
upper_bound = q3 + 1.5*iqr
outlier_mask = (sample_df['purchase_amount']< lower_bound) | (sample_df['purchase_amount']> upper_bound)

# shows the number of outlier values
print(outlier_mask.sum())

0


In [25]:
sample_df['purchase_amount'] = np.clip (sample_df['purchase_amount'], lower_bound, upper_bound) # limits the purchase amount between upper_bound and lower_bound

In [28]:
 # 5. Convert signup_date to proper datetime
sample_df["signup_date"] = pd.to_datetime(sample_df["signup_date"], errors="coerce")

In [29]:
# 6. Impute missing numeric values with median
for col in ['age', 'purchase_amount']:
    missing = sample_df[col].isna().sum()
    median_val = sample_df[col].median()
    sample_df[col] = sample_df[col].fillna(median_val)
    print(f"[Clean] Filled {missing} missing values in '{col}' with median ({median_val:.2f})")

[Clean] Filled 10 missing values in 'age' with median (47.00)
[Clean] Filled 10 missing values in 'purchase_amount' with median (157.49)


In [30]:
# 7. Handle missing categorical values
sample_df['region'] = sample_df['region'].fillna('Unknown')
sample_df['product_category'] = sample_df['product_category'].fillna('Unknown')

In [32]:
# 8. Missing satisfaction_score -> treat as NaN (kept, since imputing an opinion score is risky);but ensure dtype is consistent (nullable integer)
sample_df["satisfaction_score"] = pd.to_numeric(sample_df["satisfaction_score"], errors="coerce").astype("Int64")

In [33]:
# 9. Final dtype tidy-up
sample_df["age"] = sample_df["age"].round(0).astype(int)
sample_df["purchase_amount"] = sample_df["purchase_amount"].round(2)

In [34]:
sample_df.shape

(200, 7)

In [37]:
sample_df

,customer_name,region,age,signup_date,purchase_amount,product_category,satisfaction_score
0,Customer_1,South,47,2022-01-01 00:00:00.000000,129.64,Clothing,3
1,Customer_2,Unknown,69,2022-01-06 12:03:37.085427,101.08,Groceries,3
2,Customer_3,West,28,2022-01-12 00:07:14.170854,191.95,Groceries,4
3,Customer_4,East,66,2022-01-17 12:10:51.256281,185.99,Groceries,3
4,Customer_5,North,25,2022-01-23 00:14:28.341708,157.49,Clothing,4
...,...,...,...,...,...,...,...
195,Customer_196,South,29,2024-12-08 23:45:31.658291,185.17,Groceries,4
196,Customer_197,West,43,2024-12-14 11:49:08.743718,58.61,Home,4
197,Customer_198,North,33,2024-12-19 23:52:45.829145,318.56,Groceries,3
198,Customer_199,North,68,2024-12-25 11:56:22.914572,172.54,Home,3
